# Analytics & Data Storytelling for AI Solutions
## Hands-on: Evaluasi Chatbot Customer Service
**Applied AI Engineer Bootcamp — 40 menit**

Kasus KirimKita: pilih kandidat chatbot untuk pilot terbatas. Semua angka sintetis. Gunakan `readme.html` sebagai panduan lengkap dan `workspace/jawaban.md` untuk penjelasan.

Isi kode yang bertanda `...`, `__PIVOT__`, atau `NotImplementedError`. Jalankan sel satu per satu sesuai urutan. Output pemrograman tersimpan di folder `output`.

## 0. Persiapan
Lokal: ekstrak ZIP dahulu. Colab: buka notebook ini, lalu jalankan sel berikut dan unggah ZIP paket yang sesuai.

In [ ]:
from pathlib import Path
import sys
# Lokal: notebook dapat dibuka dari root paket atau folder workspace/solution.
# Colab: buka notebook ini dahulu, lalu unggah ZIP paket pada sel ini.
if 'google.colab' in sys.modules:
    from google.colab import files
    import zipfile
    candidates = list(Path('/content').glob('Paket_*_Analytics_AI/data/chatbot_evaluation.csv'))
    if not candidates:
        print('Unggah ZIP paket peserta atau instruktur yang sesuai.')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(zip_names) != 1:
            raise ValueError('Unggah tepat satu ZIP paket, lalu jalankan ulang sel.')
        target = Path('/content').resolve()
        with zipfile.ZipFile(zip_names[0]) as archive:
            for item in archive.infolist():
                if not (target / item.filename).resolve().is_relative_to(target):
                    raise ValueError('Path ZIP tidak valid.')
            archive.extractall(target)
        candidates = list(target.glob('Paket_*_Analytics_AI/data/chatbot_evaluation.csv'))
    if len(candidates) != 1:
        raise ValueError('Gunakan satu paket per runtime Colab agar dataset tidak ambigu.')
    ROOT = candidates[0].parent.parent
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'data/chatbot_evaluation.csv').exists()), None)
    if ROOT is None:
        raise FileNotFoundError('Buka notebook dari folder paket yang sudah diekstrak.')
print('Folder paket:', ROOT)
OUT = ROOT / 'output'
OUT.mkdir(exist_ok=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util
spec = importlib.util.spec_from_file_location('check_data', ROOT / 'starter/check_data.py')
checker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(checker)
df = pd.read_csv(ROOT / 'data/chatbot_evaluation.csv')
checker.validate_source(df)
print('Dimensi:', df.shape)
display(df.head())

## A. Metrik dan keputusan — 5 menit
Isi bagian A lembar jawaban: grain, tujuan keputusan, dua kelompok metrik, dan definisi denominator. Belum perlu menulis kode.

## B. Profiling — 5 menit
Lengkapi kode. Jelaskan mengapa ID pertanyaan muncul pada dua baris tanpa menjadi duplikasi pasangan ID–versi.

In [ ]:
# Challenge B: ganti ... dengan kolom yang benar.
# Gunakan question_id untuk menghitung pertanyaan unik.
print('Jumlah baris:', len(df))
print('Pertanyaan unik:', df[...].nunique())
print('Duplikat pasangan ID-versi:', df.duplicated([...]).sum())
display(df.groupby(['model_version', 'category']).size().rename('n'))

## C. Perhitungan — 10 menit
C1: ringkasan per versi. C2: pass rate per kategori. C3: kasus gagal. Referensi sintaks kecil tersedia pada starter/demo.ipynb.

In [ ]:
def make_summary(data):
    # Challenge C1: groupby model_version lalu hitung metrik.
    # Kolom wajib: model_version, n, pass_rate_pct, resolution_rate_pct,
    # median_latency_s, p95_latency_s, avg_cost_usd.
    # Persen menggunakan skala 0–100. P95: quantile(0.95), interpolasi linear.
    raise NotImplementedError('Lengkapi make_summary, lalu jalankan ulang sel.')
summary = make_summary(df)
display(summary)

In [ ]:
def make_category_summary(data):
    # Challenge C2: kelompokkan menurut model_version dan category.
    # Kolom wajib: model_version, category, n, pass_rate_pct.
    raise NotImplementedError('Lengkapi agregasi kategori.')
def get_failed_cases(data):
    # Challenge C3: kembalikan semua kolom hanya untuk answer_pass == 0.
    raise NotImplementedError('Lengkapi filter kegagalan.')
category_summary = make_category_summary(df)
failed_cases = get_failed_cases(df)
display(category_summary)
display(failed_cases.head())

## D. Dashboard dan ekspor — 10 menit
Lengkapi pivot lalu jalankan scaffold visual. Sumbu persen 0–100. Bandingkan metrik dengan satuan yang sama. Periksa label dan legenda sebelum menyimpulkan.

In [ ]:
# Challenge D: ubah tabel panjang menjadi kategori x versi.
# Gunakan pivot(index='category', columns='model_version', values='pass_rate_pct').
category_pivot = __PIVOT__
versions = summary.set_index('model_version').sort_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = ['#386CB0', '#FF6333']
category_pivot.plot.bar(ax=axes[0, 0], color=colors, rot=0)
axes[0, 0].set(title='Answer pass rate per kategori', ylabel='Persen (%)', xlabel='Kategori (n=50/versi)', ylim=(0, 100))
versions[['pass_rate_pct', 'resolution_rate_pct']].plot.bar(ax=axes[0, 1], color=colors, rot=0)
axes[0, 1].set(title='Kualitas dan penyelesaian', ylabel='Persen (%)', xlabel='Versi', ylim=(0, 100))
axes[0, 1].legend(['Answer pass', 'Resolved simulasi'])
versions[['median_latency_s', 'p95_latency_s']].plot.bar(ax=axes[1, 0], color=colors, rot=0)
axes[1, 0].set(title='Waktu respons', ylabel='Detik', xlabel='Versi')
axes[1, 0].legend(['Median', 'P95'])
versions['avg_cost_usd'].plot.bar(ax=axes[1, 1], color=['#386CB0', '#FF6333'], rot=0)
axes[1, 1].set(title='Rata-rata biaya pemrosesan', ylabel='USD/pertanyaan', xlabel='Versi')
fig.suptitle('Evaluasi Chatbot A/B | Data sintetis | 200 kasus per versi', fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(OUT / 'dashboard.png', dpi=160, bbox_inches='tight')
plt.show()
print('Tersimpan:', OUT / 'dashboard.png')

In [ ]:
# Jalankan setelah fungsi dan grafik selesai. Simpan presisi penuh untuk pemeriksaan.
summary.to_csv(OUT / 'summary.csv', index=False)
category_summary.to_csv(OUT / 'category_summary.csv', index=False)
failed_cases.to_csv(OUT / 'failed_cases.csv', index=False)
print('Tersimpan: summary.csv, category_summary.csv, failed_cases.csv')

## E. Pemeriksaan dan cerita — 10 menit
Jalankan pemeriksaan, isi tiga temuan berangka, trade-off, rekomendasi, dan keterbatasan pada lembar jawaban.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, str(ROOT / 'test/check_results.py')], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    print('Masih ada FAIL. Perbaiki jawaban dan ekspor ulang hasil.')

## Rekomendasi
**Tulis jawaban Anda:** tujuan, bukti, implikasi, tindakan, dan keterbatasan.

## Simpan hasil
Download notebook dan lembar jawaban. Sel berikut mengemas file output yang sudah dibuat, tanpa kunci jawaban.

In [ ]:
# Opsional: unduh hasil dari Colab sebelum menutup runtime.
import shutil
submission_dir = OUT / 'submission'
submission_dir.mkdir(exist_ok=True)
for name in ['summary.csv', 'category_summary.csv', 'failed_cases.csv', 'dashboard.png', 'checks.csv']:
    source = OUT / name
    if source.exists():
        shutil.copy2(source, submission_dir / name)
zip_path = shutil.make_archive(str(OUT / 'hasil_analisis'), 'zip', submission_dir)
print(zip_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(zip_path)
# Unduh notebook melalui File > Download > Download .ipynb dan simpan lembar jawaban terpisah.